# 01. Metadata Cache and Split Logic

This notebook focuses on the middle of the data pipeline:

```text
h5ad obs columns
  -> H5MetadataCache: categories and integer codes
  -> split_strategy.py: train / validation / test index arrays
  -> setup_dataset(): wrap split arrays into H5adSentenceDataset
```

For Replogle fine-tuning, the split function used is `split_tahoe100m(...)`, because Replogle shares the same perturb-seq holdout style.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=None):
    cur = Path(start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / "src" / "data").exists() and (p / "configs").exists():
            return p
    raise RuntimeError("Could not find PerturbDiff repo root")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PERTURB_ROOT = PROJECT_ROOT / "data" / "replogle" / "perturb_data"
RUN_DIR = PROJECT_ROOT / "runs" / "data_pipeline_splits"
WANDB_DIR = PROJECT_ROOT / "wandb"
assert PERTURB_ROOT.exists(), "Run example/replogle/00_download_data.ipynb first."

In [ ]:
import numpy as np
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

from src.common.utils import setup_loggings
from src.apps.sampling.sampling_setup import build_sampling_datamodule
from src.data.metadata_cache import GlobalH5MetadataCache
from src.data.split_strategy import split_tahoe100m

overrides = [
    "path=trixie_path",
    f"path.tmp_dir={PERTURB_ROOT}",
    f"path.diffusion.save_dir={RUN_DIR}",
    f"path.wandb.logging_dir={WANDB_DIR}",
    "data=replogle_finetune",
    "data.normalize_counts=10",
    "data.pad_length=2000",
    "data.embed_key=X_hvg",
    "model.hidden_num=[2000,512]",
    "model.input_dim=2000",
    "optimization.micro_batch_size=16",
    "data.use_cell_set=8",
]

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "configs"), version_base=None):
    cfg = compose(config_name="rawdata_diffusion_sampling", overrides=overrides)
OmegaConf.resolve(cfg)
logger = setup_loggings(cfg)
dm = build_sampling_datamodule(cfg, logger)

print("dm:", type(dm).__name__)
print("splits:", dm.all_split_names)
print("dataset_path:", cfg.data.dataset_path)

## H5MetadataCache

`H5MetadataCache` reads lightweight obs metadata once: categories, integer codes, control mask, and batch handling. It does not read full expression matrices.

In [ ]:
meta_cache = GlobalH5MetadataCache()
cache = meta_cache.get_cache(
    cfg.data.dataset_path,
    batch_col=cfg.data.perturbseq_batch_col,
    pert_col=cfg.data.pert_col,
    cell_type_key=cfg.data.cell_line_key,
)

print("n_cells:", cache.n_cells)
print("pert categories:", len(cache.pert_categories), cache.pert_categories[:10])
print("cell type categories:", len(cache.cell_type_categories), cache.cell_type_categories)
print("batch categories:", len(cache.batch_categories), cache.batch_categories[:10])
print("pert_codes:", cache.pert_codes.shape, cache.pert_codes.min(), cache.pert_codes.max())
print("cell_type_codes:", cache.cell_type_codes.shape, cache.cell_type_codes.min(), cache.cell_type_codes.max())
print("batch_codes:", cache.batch_codes.shape, cache.batch_codes.min(), cache.batch_codes.max())

## Holdout Definitions

The Replogle config holds out a cell line and lists of perturbations. `split_tahoe100m()` intersects those conditions to make validation and test indices.

In [ ]:
key_info = dm.key_info_list[0]
print("holdout_celltype:", key_info["holdout_celltype"])
print("validation holdout pert count:", len(key_info["holdout_pert"]["validation"]))
print("test holdout pert count:", len(key_info["holdout_pert"]["test"]))
print("validation examples:", key_info["holdout_pert"]["validation"][:10])
print("test examples:", key_info["holdout_pert"]["test"][:10])

In [ ]:
split_indices = split_tahoe100m(dm, cache, "pert", key_info, "holdout_pert", "random_Perturbseq")
for split, idx in split_indices.items():
    print(split, idx.shape, idx[:10])

print("validation/test overlap:", np.isin(split_indices["validation"], split_indices["test"]).sum())
print("train/validation overlap:", np.isin(split_indices["train"], split_indices["validation"]).sum())
print("train/test overlap:", np.isin(split_indices["train"], split_indices["test"]).sum())

## What Is in Each Split?

The split arrays are local row indices into the `.h5ad`. They still include control cells in validation/test when `split_control=false`, because controls are used for mapping/conditioning.

In [ ]:
def summarize_indices(name, indices):
    pert_names = cache.pert_categories[cache.pert_codes[indices]]
    ct_names = cache.cell_type_categories[cache.cell_type_codes[indices]]
    batch_names = cache.batch_categories[cache.batch_codes[indices]]
    print("\n", name)
    print("  n:", len(indices))
    print("  unique perts:", len(set(pert_names)), list(sorted(set(pert_names)))[:10])
    print("  unique cell types:", sorted(set(ct_names)))
    print("  unique batches:", len(set(batch_names)), list(sorted(set(batch_names)))[:10])
    print("  control cells:", int((pert_names == cfg.data.control_pert).sum()))

for split, idx in split_indices.items():
    summarize_indices(split, idx)

## `setup_dataset()` Materializes Dataset Objects

`dm.setup()` only built global category dictionaries. `dm.setup_dataset()` builds:

- `train_indices`, `validation_indices`, `test_indices`
- `train_dataset`, `validation_dataset`, `test_dataset`
- each dataset owns an `H5Store`, selected gene mask, control/perturb split, and grouped indices for cell-set sampling.

In [ ]:
dm.setup_dataset()
for split in ["train", *dm.all_split_names]:
    dataset = getattr(dm, f"{split}_dataset")
    indices = getattr(dm, f"{split}_indices")
    num_cell = getattr(dm, f"{split}_num_cell")
    print("\n", split)
    print("  dataset class:", type(dataset).__name__)
    print("  len(dataset):", len(dataset))
    print("  index dict keys:", list(indices.keys()))
    print("  num_cell:", num_cell)
    print("  control_type:", dataset.control_type)
    print("  grouped_pert_data_indices groups:", {k: len(v) for k, v in dataset.grouped_pert_data_indices.items()})